# 10.1 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/10-deployment-mlops/01-saving-and-reproducibility/solutions.ipynb)

In [1]:
# --- course files: found locally, downloaded on Colab -------------------------
import sys, urllib.request
from pathlib import Path
REPO_RAW = "https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/"

def repo_file(rel):
    """Return a local path to a file in this repo, downloading it if needed."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / rel).exists():
            return parent / rel
    dest = Path("course_files") / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists():
        urllib.request.urlretrieve(REPO_RAW + rel, dest)
    return dest

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
repo_file("data/download.py")                                  # churn_model finds the data through it
sys.path.insert(0, str(repo_file("lib/churn_model.py").parent))
import churn_model as cm

bundle = cm.load_bundle()          # trains and saves models/churn_bundle.joblib the first time
ROOT = cm.repo_root()
print(f"Model trained {bundle['trained']}, test AUC {bundle['metrics']['test_auc']}, "
      f"threshold {bundle['threshold']}")

Model trained 2026-10-28, test AUC 0.741, threshold 0.07


## Exercise 1

In [2]:
scaler, logreg = bundle["pipeline"][0], bundle["pipeline"][-1]
x = np.array([10, 2000, 60, 300, 20], dtype=float)
z = (x - scaler.mean_) / scaler.scale_
contrib = logreg.coef_[0] * z
logit = logreg.intercept_[0] + contrib.sum()
print(pd.Series(contrib, index=bundle["features"]).round(3))
print(f"logit {logit:.3f}  →  p = {1 / (1 + np.exp(-logit)):.4f}")
print(cm.score(bundle, pd.DataFrame([dict(zip(bundle["features"], x))])).iloc[0].to_dict())

orders            -1.267
spend              0.005
products          -0.084
tenure_days       -0.274
days_since_last   -0.408
dtype: float64
logit -2.504  →  p = 0.0756
{'churn_probability': 0.0756, 'voucher': True}


Ten orders is about 1.3 standard deviations above average and pulls the log-odds down hard; so does recency of 20 days. This loyal customer still sits above the 0.07 threshold: at a £5 voucher almost everyone gets one (05.10).

## Exercise 2

In [3]:
import argparse
def score_file(path, threshold=None):
    df = pd.read_csv(path)
    values = df[bundle["features"]]
    good = df[~(values.isna().any(axis=1) | (values < 0).any(axis=1))]
    p = bundle["pipeline"].predict_proba(good[bundle["features"]])[:, 1]
    t = bundle["threshold"] if threshold is None else threshold
    return int((p >= t).sum()), len(good)

csv_in = ROOT / "models" / "customers_to_score.csv"
if not csv_in.exists():                      # made in the lesson; recreate if needed
    cust = cm.build_customer_table(cm.load_sales())
    cust.sample(300, random_state=1)[cm.FEATURES].to_csv(csv_in, index=False)
for t in [None, 0.35]:
    n, total = score_file(csv_in, t)
    print(f"threshold {t or bundle['threshold']}: {n} vouchers of {total}")

threshold 0.07: 277 vouchers of 298
threshold 0.35: 201 vouchers of 298


In the script itself, add `parser.add_argument("--threshold", type=float)` and use `args.threshold or bundle["threshold"]` when deciding vouchers. A stricter threshold of 0.35 (the cheapest one for a £20 voucher, 10.7) cuts the mailing from 277 to 201 customers without touching the model: the policy, not the model, decides most of the cost.

## Exercise 3

In [4]:
import joblib, sklearn
from sklearn.metrics import roc_auc_score
REQUIRED = {"pipeline", "features", "threshold", "metrics", "trained", "sklearn_version"}

def check_bundle(path, table):
    b = joblib.load(path)
    missing = REQUIRED - set(b)
    print("missing keys     :", sorted(missing) or "none")
    print("version matches  :", b.get("sklearn_version") == sklearn.__version__)
    auc = roc_auc_score(table["churned"], b["pipeline"].predict_proba(table[b["features"]])[:, 1])
    print(f"AUC on this table: {auc:.3f}", "(WARNING: below 0.70)" if auc < 0.70 else "")

check_bundle(cm.MODEL_PATH, cm.build_customer_table(cm.load_sales()))

missing keys     : none
version matches  : True
AUC on this table: 0.743 


The AUC here is on all 3,317 customers, most of whom the model trained on, so it is slightly optimistic. In production you would run this on the newest customers whose labels have just arrived (10.6).